# Dataset Fixes — Philippine Educational System

This notebook fixes five problems in the final dataset so that the data matches what the paper describes. Run it **once**, before re-running the similarity, topic modeling, and sentiment notebooks.

| # | Problem | What this notebook does |
|---|---|---|
| 1 | The `lemmas` column contains broken word stems (*deped → dep*, *including → includ*, *previous → previou*) | Rebuilds `lemmas` from `tokens` with the WordNet lemmatizer. Words WordNet doesn't know (Tagalog words, names like *DepEd*) are left unchanged. |
| 2 | The paper lists 7 search keywords, but scraping used 91 | Prints the full keyword list from the raw data and the keywords that appear in the final dataset. |
| 3 | Some records are unrelated to education (Graham's number, phone ads, foreign politics) | Replaces each off-topic record with an on-topic record **of the same type, in the same split**, taken from the cleaned records that were not used. |
| 4 | The record counts in the paper disagree (17,000+ / 17,924) | Counts the raw combined file by platform and record type. |
| 5 | The paper doesn't say how the 90/10 split was made | Verifies that the split is **stratified by record type** and that it still is after the fixes. |

**Files to upload to `/content/`:** `final_combined_data.csv`, `final_cleaned_combined_data.csv`, `final_training_data.csv`, `final_unseen_data.csv`

**Output:** fixed `final_training_data.csv` and `final_unseen_data.csv` (same names, saved in the same folder, so the other notebooks work unchanged), a log of every replaced record, and `records_to_label.csv` (the records that still need a manual sentiment label).

**Local run:** the notebook also works from the repo folder; it reads and writes the files in `datasets/`.

## Step 1 — Load the files

In [ ]:
import glob, os

# Looks in datasets/ (running locally from the repo) and /content/ (Colab uploads).
# Also finds Colab-renamed copies like "final_training_data (1).csv".
DATA_DIRS = ["datasets", "/content/datasets", "/content", "."]

def find_file(name, required=True):
    stem, ext = os.path.splitext(name)
    for d in DATA_DIRS:
        exact = os.path.join(d, name)
        if os.path.exists(exact):
            return exact
        hits = sorted(glob.glob(os.path.join(d, stem + " (*)" + ext)))
        if hits:
            return hits[-1]
    if required:
        raise FileNotFoundError(f"Upload {name} to /content/ (Colab) or keep it in datasets/ (local).")
    return None

import re, csv
import pandas as pd
import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
from nltk.stem import WordNetLemmatizer

pd.set_option("display.max_colwidth", 120)

TRAIN_PATH  = find_file("final_training_data.csv")
UNSEEN_PATH = find_file("final_unseen_data.csv")
OUT_DIR     = os.path.dirname(TRAIN_PATH)   # fixed files are saved next to the originals

raw = pd.read_csv(
    find_file("final_combined_data.csv"),
    engine="python",
    on_bad_lines="skip",
    encoding_errors="replace",
)
cleaned  = pd.read_csv(find_file("final_cleaned_combined_data.csv"))
train    = pd.read_csv(TRAIN_PATH)
unseen   = pd.read_csv(UNSEEN_PATH)

print("Raw combined records:   ", len(raw))
print("Cleaned records:        ", len(cleaned))
print("Training records:       ", len(train))
print("Unseen records:         ", len(unseen))

## Step 2 — Fix #4: Count the raw records
The paper should use the count from the actual combined file. This table gives the exact breakdown.

In [ ]:
raw_counts = raw.groupby(["platform", "record_type"]).size().rename("records").reset_index()
display(raw_counts)
print("TOTAL raw records after merging and removing duplicates:", len(raw))

,platform,record_type,records
0,reddit,reddit_comment,228
1,reddit,reddit_post,200
2,tiktok,tiktok_video,53
3,web,article,162
4,youtube,youtube_comment,17305


TOTAL raw records after merging and removing duplicates: 17948


## Step 3 — Fix #2: Search keywords
**All keywords used during scraping** (from the raw file) and **the keywords that appear in the final 200 records**. Articles have no keyword because they were collected from news-site tag pages, not by keyword search.

In [ ]:
kw_by_type = raw.groupby("record_type")["keyword"].nunique()
print("Number of distinct search keywords used during scraping:", raw["keyword"].nunique())
display(kw_by_type.rename("distinct keywords").to_frame())

all_keywords = (raw["keyword"].dropna().str.strip().value_counts()
                .rename_axis("keyword").reset_index(name="raw_records"))
print("\nAll search keywords (raw data):")
display(all_keywords)

final = pd.concat([train, unseen])
final_keywords = final["keyword"].value_counts().rename_axis("keyword").reset_index(name="final_records")
print(f"\nKeywords present in the final {len(final)} records: {len(final_keywords)} "
      f"(+ {final['keyword'].isna().sum()} articles with no keyword)")
display(final_keywords)

Number of distinct search keywords used during scraping: 91


,distinct keywords
record_type,
article,0
reddit_comment,1
reddit_post,4
tiktok_video,6
youtube_comment,89



All search keywords (raw data):


,keyword,raw_records
0,school supplies Philippines,727
1,learning Philippines,672
2,Philippine students performance,627
3,Filipino teachers,618
4,Philippine education,616
...,...,...
86,learning crisis Philippines,9
87,educational system Philippines,5
88,education crisis Philippines,5
89,education access Philippines,5



Keywords present in the final 200 records: 27 (+ 30 articles with no keyword)


,keyword,final_records
0,philippine education,59
1,digital education philippines,11
2,education assessment philippines,9
3,deped philippines,8
4,online learning philippines,7
5,blended learning philippines,6
6,college tuition philippines,6
7,philippines teachers,5
8,school facilities philippines,5
9,education curriculum philippines,5


## Step 4 — Fix #3: Replace off-topic records
Each record below was read manually and judged **unrelated to the Philippine educational system**. Each one is replaced with an on-topic record:
- of the **same record type** (so the dataset keeps 80 / 55 / 30 / 20 / 15 records per type), and
- placed in the **same split** (so the 90/10 stratification is unchanged),
- taken from the **cleaned records that were never used** in the training or unseen sets (so it went through the same preprocessing).

Short reactions like *"thank you po"* or *"galing"* are **kept on purpose**: they're on-topic replies to education videos and carry sentiment, which the sentiment analysis needs. They're automatically excluded from topic modeling because they have too few words.

Two TikTok videos (`data00611`, sleep hours by age; `data00602`, a BPO job ad) are kept because the reserve pool had no more on-topic TikTok videos. Report them as a limitation.

In [ ]:
# off-topic record_id: (reason, replacement record_id from the unused cleaned records)
REPLACEMENTS = {
    # Articles
    "data00093": ("Book launch column, not about education",                   "data00044"),
    "data00094": ("Opinion on one-China policy",                               "data00063"),
    "data00095": ("Opinion on a US–Venezuela oil deal",                        "data00111"),
    "data00006": ("Impeachment trial news",                                    "data00033"),
    "data00096": ("Obituary column for a national artist",                     "data00061"),   # unseen set
    # TikTok videos
    "data00637": ("Graham's number (mathematics trivia)",                      "data00613"),
    "data00632": ("Phone advertisement",                                       "data00592"),
    "data00631": ("Government bond investment ad",                             "data00630"),
    "data00607": ("Business price increases",                                  "data00596"),
    "data00641": ("Farm work wages",                                           "data00593"),
    "data00642": ("Vlog request from friends",                                 "data00628"),
    # YouTube comments
    "data16753": ("COVID early-treatment debate",                              "data01542"),
    "data12413": ("Comment about an unrelated violent crime",                  "data01580"),
    "data17578": ("Comment about appearance",                                  "data10978"),
    "data11357": ("Unrelated question about colors",                           "data09888"),
    "data11754": ("Politicians becoming millionaires",                         "data02420"),
    "data11042": ("Election candidates' promises",                             "data10947"),
    "data11359": ("Typhoons and political leaders",                            "data00928"),
    # Reddit comments
    "data00293": ("Senator's appearance",                                      "data00210"),
    "data00262": ("Reaction about workplace subreddits",                       "data00342"),
}

used_ids = set(train["record_id"]) | set(unseen["record_id"])
reserve = cleaned[~cleaned["record_id"].isin(used_ids)].set_index("record_id")

log = []
def apply_replacements(split_df, split_name):
    split_df = split_df.copy()
    for old_id, (reason, new_id) in REPLACEMENTS.items():
        if old_id not in set(split_df["record_id"]):
            continue
        old = split_df.loc[split_df["record_id"] == old_id].iloc[0]
        new = reserve.loc[new_id]
        # safety checks: replacement must exist, be unused, and have the same record type
        assert new["record_type"] == old["record_type"], f"{new_id} is not a {old['record_type']}"
        new_row = new.reindex(split_df.columns.drop("record_id")).to_dict()
        new_row["record_id"] = new_id
        split_df.loc[split_df["record_id"] == old_id, list(new_row)] = list(new_row.values())
        log.append({"split": split_name, "record_type": old["record_type"],
                    "removed_id": old_id, "reason": reason,
                    "removed_text": str(old["text"] if pd.notna(old["text"]) else old["title"])[:150],
                    "added_id": new_id,
                    "added_text": str(new["text"] if pd.notna(new["text"]) else new["title"])[:150]})
    return split_df

train_fixed  = apply_replacements(train,  "training")
unseen_fixed = apply_replacements(unseen, "unseen")

replacement_log = pd.DataFrame(log)
print(f"Records replaced: {len(replacement_log)} "
      f"({(replacement_log['split'] == 'training').sum()} training, {(replacement_log['split'] == 'unseen').sum()} unseen)")
display(replacement_log[["split", "record_type", "removed_id", "reason", "added_id", "added_text"]])

# Nothing duplicated, nothing lost
all_ids = pd.concat([train_fixed["record_id"], unseen_fixed["record_id"]])
assert all_ids.is_unique, "A record appears twice"
assert len(train_fixed) == len(train) and len(unseen_fixed) == len(unseen)
print("Checks passed: no duplicates, split sizes unchanged.")

Records replaced: 20 (19 training, 1 unseen)


,split,record_type,removed_id,reason,added_id,added_text
0,training,article,data00093,"Book launch column, not about education",data00044,at a glance - the philippines improved its reading mathematics and science rankings in the programme for internation...
1,training,article,data00094,Opinion on one-China policy,data00063,at a glance - the philippines showed significant improvement in the pisa results with increases in reading math and ...
2,training,article,data00095,Opinion on a US–Venezuela oil deal,data00111,was it really a success education secretary leonor briones declared the opening of another school year amid the covi...
3,training,article,data00006,Impeachment trial news,data00033,at a glance - in of filipino students reported experiencing bullying according to the programme for international st...
4,training,tiktok_video,data00637,Graham's number (mathematics trivia),data00613,sa ilang remote at last-mile schools access to electricity can change how teachers teach and how students learn sa t...
5,training,tiktok_video,data00632,Phone advertisement,data00592,sa national expenditure program o proposed budget para sa fy na inihain sa kongreso bilyon ang inilaan para sa deped...
6,training,tiktok_video,data00631,Government bond investment ad,data00630,list of college degrees in the philippines if you want to ai proof youself according to forbes
7,training,tiktok_video,data00607,Business price increases,data00596,ano ang average last term aabot kaya ng ang average mo ngayung 2nd term goodluck 2ndterm privateschool generalaverag...
8,training,tiktok_video,data00641,Farm work wages,data00593,pabor ba kayo ba gawing 6am to am lang ang klase ng mga bata dahil sa sobrang init dulot ng super el niño teacher te...
9,training,tiktok_video,data00642,Vlog request from friends,data00628,kung isa ka sa mga parent na maagang na-expose sa gadgets ang anak this is for you ud83d udcf1 ud83d udc76 alamin ku...


Checks passed: no duplicates, split sizes unchanged.


## Step 5 — Fix #1: Rebuild the `lemmas` column
The original suffix rules cut words into non-words (*deped → dep*, *including → includ*, *campus → campu*). We rebuild `lemmas` from the `tokens` column using the **WordNet lemmatizer**:
- try the word as a **noun** (*students → student*, *children → child*), then as a **verb** (*taught → teach*, *including → include*);
- WordNet only changes a word if the result is a real English word, so **Tagalog words and names stay unchanged** (*deped*, *guro*, *pilipinas*);
- words of 3 letters or fewer are left as they are, to avoid errors like *us → u*.

In [ ]:
lemmatizer = WordNetLemmatizer()

def lemmatize_token(tok):
    if len(tok) <= 3 or not tok.isalpha():
        return tok
    noun = lemmatizer.lemmatize(tok, pos="n")
    if noun != tok:
        return noun
    return lemmatizer.lemmatize(tok, pos="v")

def rebuild_lemmas(tokens):
    if pd.isna(tokens):
        return tokens
    return " ".join(lemmatize_token(t) for t in str(tokens).split())

for frame in (train_fixed, unseen_fixed):
    frame["lemmas_old"] = frame["lemmas"]
    frame["lemmas"] = frame["tokens"].apply(rebuild_lemmas)

# Before / after examples
examples = ["deped", "including", "previous", "campus", "learning", "taught", "students", "children",
            "teachers", "pilipinas", "guro", "universities", "focus", "studies"]
display(pd.DataFrame({"word": examples, "new lemma": [lemmatize_token(w) for w in examples]}))

changed = (train_fixed["lemmas"] != train_fixed["lemmas_old"]).sum()
print(f"Training records whose lemmas changed: {changed} of {len(train_fixed)}")
display(train_fixed.loc[train_fixed["lemmas"] != train_fixed["lemmas_old"], ["tokens", "lemmas_old", "lemmas"]].head(5))

for frame in (train_fixed, unseen_fixed):
    frame.drop(columns="lemmas_old", inplace=True)

,word,new lemma
0,deped,deped
1,including,include
2,previous,previous
3,campus,campus
4,learning,learn
5,taught,teach
6,students,student
7,children,child
8,teachers,teacher
9,pilipinas,pilipinas


Training records whose lemmas changed: 152 of 180


,tokens,lemmas_old,lemmas
0,deped launch digital safety campaign banga school shooting p99-b budget amendments questionable director filmmaker t...,dep launch digital safety campaign banga school shoot p99-b budget amendment questionable director filmmaker toronto...,deped launch digital safety campaign banga school shoot p99-b budget amendment questionable director filmmaker toron...
2,pabor ba kayo ba gawing 6am lang klase bata sobrang init dulot super el niño teac pabor ba kayo ba gawing 6am lang k...,pabor ba kayo ba gaw 6am lang klase bata sobrang init dulot super el niño teac pabor ba kayo ba gaw 6am lang klase b...,pabor ba kayo ba gawing 6am lang klase bata sobrang init dulot super el niño teac pabor ba kayo ba gawing 6am lang k...
3,angara pinangunahan mass oath-taking bagong promoted guro davao region lungsod davao oktubre pinangunahan education ...,angara pinangunahan mass oath-tak bagong promot guro davao region lungsod davao oktubre pinangunahan education secre...,angara pinangunahan mass oath-taking bagong promote guro davao region lungsod davao oktubre pinangunahan education s...
4,nakakalungkot talaga lagay edukasyon pilipinas kamay ganitong guro nagkakamali teachers natin pwedeng mabait lang ba...,nakakalungkot talaga lagay edukasyon pilipina kamay ganitong guro nagkakamali teacher natin pwedeng mabait lang bata...,nakakalungkot talaga lagay edukasyon pilipinas kamay ganitong guro nagkakamali teacher natin pwedeng mabait lang bat...
5,deped approved vacation leave,dep approv vacation leave,deped approve vacation leave


## Step 6 — Fix #5: Verify the stratified 90/10 split
The split was made so that **each record type is split 90/10 separately** (stratified sampling). Types whose counts don't divide evenly are rounded (55 Reddit comments → 50/5, 15 Reddit posts → 13/2). This keeps the mix of YouTube, Reddit, article, and TikTok records the same in both sets. The table checks that this still holds after the replacements.

For reference, this is how a stratified split is created with scikit-learn:
```python
from sklearn.model_selection import train_test_split
train_df, unseen_df = train_test_split(final_df, test_size=0.10,
                                       stratify=final_df["record_type"], random_state=42)
```

In [ ]:
strat = pd.DataFrame({
    "total": pd.concat([train_fixed, unseen_fixed])["record_type"].value_counts(),
    "training": train_fixed["record_type"].value_counts(),
    "unseen": unseen_fixed["record_type"].value_counts(),
})
strat["training_%"] = (strat["training"] / strat["total"] * 100).round(1)
strat["unseen_%"] = (strat["unseen"] / strat["total"] * 100).round(1)
strat.loc["TOTAL"] = strat[["total", "training", "unseen"]].sum().tolist() + [
    round(len(train_fixed) / (len(train_fixed) + len(unseen_fixed)) * 100, 1),
    round(len(unseen_fixed) / (len(train_fixed) + len(unseen_fixed)) * 100, 1)]
display(strat)
# A record type can only be split exactly 90/10 if its count divides evenly
# (e.g. 55 Reddit comments -> 49.5 / 5.5), so we allow a difference of at most 1 record from rounding.
by_type = strat.drop(index="TOTAL")
assert ((by_type["training"] - by_type["total"] * 0.9).abs() <= 1).all(), "A record type is not split 90/10"
print("Every record type is split 90/10 (within 1 record of rounding); overall split is exactly 180/20.")

,total,training,unseen,training_%,unseen_%
record_type,,,,,
article,30.0,27.0,3.0,90.0,10.0
reddit_comment,55.0,50.0,5.0,90.9,9.1
reddit_post,15.0,13.0,2.0,86.7,13.3
tiktok_video,20.0,18.0,2.0,90.0,10.0
youtube_comment,80.0,72.0,8.0,90.0,10.0
TOTAL,200.0,180.0,20.0,90.0,10.0


Every record type is split 90/10 (within 1 record of rounding); overall split is exactly 180/20.


## Step 7 — Check the manual labels
The rule-based sentiment notebook reads its ground truth from `final_manual_labels.csv` (column `manual_label`: `1` positive, `0` neutral, `-1` negative), matched by `record_id`. After the replacements, some records in the fixed files have no label yet, and the unseen set has never been labeled. This step lists them in `records_to_label.csv` so they can be labeled and **added to `final_manual_labels.csv`**.

In [ ]:
labels_path = find_file("final_manual_labels.csv", required=False)
labels = pd.read_csv(labels_path) if labels_path else pd.DataFrame(columns=["record_id", "manual_label"])
labeled_ids = set(labels.dropna(subset=["manual_label"])["record_id"])

def text_of(row):
    return row["text"] if pd.notna(row["text"]) and str(row["text"]).strip() else row["title"]

to_label = pd.concat([train_fixed.assign(split="training"), unseen_fixed.assign(split="unseen")])
to_label = to_label[~to_label["record_id"].isin(labeled_ids)].copy()
to_label["text"] = to_label.apply(text_of, axis=1)
to_label = to_label[["record_id", "split", "record_type", "platform", "text"]]
to_label["manual_label"] = ""

final_ids = set(train_fixed["record_id"]) | set(unseen_fixed["record_id"])
unused_labels = labels[~labels["record_id"].isin(final_ids)]

for split in ["training", "unseen"]:
    total = len(train_fixed) if split == "training" else len(unseen_fixed)
    missing = (to_label["split"] == split).sum()
    print(f"{split:9s}: {total - missing} of {total} records labeled, {missing} still need a label")
print(f"Labels for records that are no longer in the dataset (ignored): {len(unused_labels)}")
display(to_label)

## Step 8 — Save the fixed files

In [ ]:
def out(name):
    return os.path.join(OUT_DIR, name)

train_fixed.to_csv(out("final_training_data.csv"), index=False)
unseen_fixed.to_csv(out("final_unseen_data.csv"), index=False)
if len(replacement_log) or not os.path.exists(out("replaced_records_log.csv")):
    replacement_log.to_csv(out("replaced_records_log.csv"), index=False)
else:
    print("Nothing new was replaced (already fixed); keeping the existing replaced_records_log.csv")
to_label.to_csv(out("records_to_label.csv"), index=False)
all_keywords.to_csv(out("search_keywords_raw.csv"), index=False)
raw_counts.to_csv(out("raw_record_counts.csv"), index=False)
print(f"Saved in {os.path.abspath(OUT_DIR)}:")
print(" - final_training_data.csv\n - final_unseen_data.csv")
print(" - replaced_records_log.csv (for the appendix)")
print(" - records_to_label.csv (label these, then add them to final_manual_labels.csv)")
print(" - search_keywords_raw.csv and raw_record_counts.csv (for the Methods section)")

try:
    from google.colab import files
    for f in ["final_training_data.csv", "final_unseen_data.csv", "replaced_records_log.csv",
              "records_to_label.csv", "search_keywords_raw.csv", "raw_record_counts.csv"]:
        files.download(out(f))
except ImportError:
    pass

## After running this notebook
1. **Manual labels:** open `records_to_label.csv`, fill in `manual_label` (`1` positive, `0` neutral, `-1` negative), and add those rows to `final_manual_labels.csv`. The unseen records must be labeled too, or the final test on the 10% cannot be scored.
2. **Re-run in this order:** sentiment (`Sentimental_Analysis.ipynb`) → similarity → topic modeling, using the fixed CSV files.
3. **Do not re-split.** The fixed files keep the original split, so no record moved between training and unseen. Running this notebook again is safe: records that were already replaced are skipped.